Шумилова А.А. М8О-407Б-21

# Выбор датасета

Мной был выбран датасет "CIFAR-10".

*Описание*:
- 60,000 изображений размером 32x32 пикселя (3 канала).
- 10 классов: `airplane`, `automobile`, `bird`, `cat`, `deer`, `dog`, `frog`, `horse`, `ship`, `truck`.
- Разделён на 50,000 обучающих и 10,000 тестовых примеров.

*Обоснование*:
- Уже встроен в torchvision.datasets, легко загрузить.
- Маленький по размеру, быстро обучается даже на CPU, а на GPU — мгновенно.
- Часто используется как бенчмарк для CV-моделей.
- Подходит как для сверточных, так и для трансформерных моделей.

*Задача классификации*: классификация объекта на изображении в один из 10 классов.

# Метрики

- Accuracy (доля правильных предсказаний): подходит, т.к. классы сбалансированы (примерно по 6,000 изображений на класс), даёт быструю общую оценку качества модели.

- F1-Score: показывает качество предсказаний по всем классам, особенно при анализе ошибок.


## Лабораторная работа №6

In [ ]:
!pip install torchmetrics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 961.5/961.5 kB 25.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 90.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 85.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 48.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 13.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 99.9 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstall

### Импорт библиотек

In [ ]:
import torch
import torchvision
import torchvision.transforms as transforms
import torch.nn as nn
import torch.optim as optim
from torchmetrics.classification import MulticlassAccuracy, MulticlassF1Score
from tqdm import tqdm

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Используемое устройство:", device)

Используемое устройство: cuda


### Преобразования и загрузка CIFAR-10

In [ ]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

trainset = torchvision.datasets.CIFAR10(root='./data', train=True,
                                        download=True, transform=transform)
trainloader = torch.utils.data.DataLoader(trainset, batch_size=64,
                                          shuffle=True, num_workers=2)

testset = torchvision.datasets.CIFAR10(root='./data', train=False,
                                       download=True, transform=transform)
testloader = torch.utils.data.DataLoader(testset, batch_size=64,
                                         shuffle=False, num_workers=2)

classes = trainset.classes

### Обучение сверточной модели (ResNet18)

In [ ]:
from torchvision.models import resnet18

model_resnet = resnet18(pretrained=True)
model_resnet.fc = nn.Linear(model_resnet.fc.in_features, 10)
model_resnet = model_resnet.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model_resnet.parameters(), lr=0.001)

/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Функция обучения

In [ ]:
def train_model(model, trainloader, criterion, optimizer, epochs=3):
    model.train()
    for epoch in range(epochs):
        running_loss = 0.0
        for inputs, labels in tqdm(trainloader):
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()

            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()

        print(f"\nЭпоха {epoch + 1}, Потери: {running_loss / len(trainloader):.4f}")

Обучение ResNet

In [ ]:
train_model(model_resnet, trainloader, criterion, optimizer)

100%|██████████| 782/782 [02:36<00:00,  5.00it/s]



Эпоха 1, Потери: 0.2430


100%|██████████| 782/782 [02:37<00:00,  4.97it/s]



Эпоха 2, Потери: 0.2417


100%|██████████| 782/782 [02:36<00:00,  5.01it/s]


Эпоха 3, Потери: 0.2419


### Обучение трансформера (deit)



Обучение deit

In [ ]:
!pip install timm

In [ ]:
import timm

In [ ]:
model = timm.create_model('deit_tiny_patch16_224', pretrained=True, num_classes=10)
model = model.to(device)

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model.safetensors:   0%|          | 0.00/22.9M [00:00<?, ?B/s]

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=3e-4)

def train_model(model, trainloader, epochs=3):
    model.train()
    for epoch in range(epochs):
        running_loss = 0.0
        for inputs, labels in tqdm(trainloader):
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()
        print(f"\nЭпоха {epoch + 1}, Потери: {running_loss / len(trainloader):.4f}")

In [ ]:
train_model(model, trainloader)

100%|██████████| 782/782 [03:05<00:00,  4.21it/s]



Эпоха 1, Потери: 0.3291


100%|██████████| 782/782 [03:05<00:00,  4.21it/s]



Эпоха 2, Потери: 0.2056


100%|██████████| 782/782 [03:05<00:00,  4.21it/s]


Эпоха 3, Потери: 0.1587


### Оценка по метрикам

In [ ]:
def evaluate_model_resnet(model, testloader):
    model.eval()
    all_preds = []
    all_labels = []
    with torch.no_grad():
        for inputs, labels in testloader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            preds = torch.argmax(outputs, 1)

            all_preds.append(preds.cpu())
            all_labels.append(labels.cpu())

    all_preds = torch.cat(all_preds)
    all_labels = torch.cat(all_labels)

    acc = MulticlassAccuracy(num_classes=10, average='macro')(all_preds, all_labels)
    f1 = MulticlassF1Score(num_classes=10, average='macro')(all_preds, all_labels)
    print(f"Accuracy: {acc:.4f}, Macro F1-score: {f1:.4f}")

In [ ]:
from sklearn.metrics import accuracy_score, f1_score

def evaluate_model_deit(model, testloader):
    model.eval()
    y_true, y_pred = [], []
    with torch.no_grad():
        for inputs, labels in testloader:
            inputs = inputs.to(device)
            outputs = model(inputs)
            _, predicted = torch.max(outputs, 1)
            y_true.extend(labels.cpu().numpy())
            y_pred.extend(predicted.cpu().numpy())

    acc = accuracy_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred, average='weighted')
    print(f"\nAccuracy: {acc:.4f}")
    print(f"F1 Score (weighted): {f1:.4f}")

Оценка ResNet

In [ ]:
evaluate_model_resnet(model_resnet, testloader)

Accuracy: 0.8763, Macro F1-score: 0.8766


Оценка ViT

In [ ]:
evaluate_model_deit(model, testloader)


Accuracy: 0.9223
F1 Score (weighted): 0.9218


### Улучшение бейзлайна

Добавим аугментации данных при обучении моделей.

Сейчас используется только Resize и Normalize. Мы можем добавить:
- RandomHorizontalFlip: поворачивает изображение по горизонтали, что помогает избежать переобучения.
- RandomCrop: обрезка с последующим ресайзом усиливает устойчивость модели.
- ColorJitter: случайно меняет яркость, контраст и насыщенность.
- RandomRotation: немного поворачивает изображения, имитируя реальную вариацию.

Обновим трансформации

In [ ]:
from torch.utils.data import DataLoader

train_transform_aug = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomCrop(32, padding=4),
    transforms.Resize((224, 224)),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

# Оставим тестовую трансформацию как есть
test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

# Переинициализируем датасеты и лоадеры
trainset_aug = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=train_transform_aug)
testset = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=test_transform)

trainloader_aug = DataLoader(trainset_aug, batch_size=64, shuffle=True)
testloader = DataLoader(testset, batch_size=64, shuffle=False)

Обучение Resnet

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model_resnet.parameters(), lr=0.001)

def train_model(model, trainloader, criterion, optimizer, epochs=3):
    model.train()
    for epoch in range(epochs):
        running_loss = 0.0
        for inputs, labels in tqdm(trainloader):
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()

            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()

        print(f"\nЭпоха {epoch + 1}, Потери: {running_loss / len(trainloader):.4f}")

train_model(model_resnet, trainloader_aug, criterion, optimizer)

100%|██████████| 782/782 [05:21<00:00,  2.43it/s]



Эпоха 1, Потери: 0.4117


100%|██████████| 782/782 [05:20<00:00,  2.44it/s]



Эпоха 2, Потери: 0.3303


100%|██████████| 782/782 [05:18<00:00,  2.45it/s]


Эпоха 3, Потери: 0.2800


Обучение deit

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=3e-4)

train_model(model, trainloader_aug, criterion, optimizer)

100%|██████████| 782/782 [05:45<00:00,  2.26it/s]



Эпоха 1, Потери: 0.2446


100%|██████████| 782/782 [05:46<00:00,  2.25it/s]



Эпоха 2, Потери: 0.2089


100%|██████████| 782/782 [05:45<00:00,  2.26it/s]


Эпоха 3, Потери: 0.1825


Оценка ResNet

In [ ]:
evaluate_model_resnet(model_resnet, testloader)

Accuracy: 0.8964, Macro F1-score: 0.8972


Оценка ViT

In [ ]:
evaluate_model_deit(model, testloader)


Accuracy: 0.9222
F1 Score (weighted): 0.9224


### Сравнение результатов

| Модель     | Accuracy (до) | F1 (до)    | Accuracy (после) | F1 (после)  |
|------------|---------------|------------|------------------|-------------|
| ResNet18   | 0.8763        | 0.8766     | **0.8964**       | **0.8972**  |
| DeiT-tiny  | 0.9223        | 0.9218     | **0.9222**       | **0.9224**  |


### Вывод

Аугментации данных — простой и эффективный способ улучшить качество ResNet18.

Они помогают предотвратить переобучение и обучить более устойчивую модель на относительно небольшом датасете, как CIFAR-10.

DeiT уже показывает высокий уровень качества без аугментаций.

Его архитектура и предобученность дают сильный старт. Аугментации дали лишь микроскопическое улучшение.

Вывод:

- Для простых моделей, таких как ResNet18, улучшения бейзлайна через аугментации — высокоэффективны.

- Для более мощных моделей, таких как DeiT, дальнейшее улучшение стоит искать в более продвинутом тюнинге: подбор learning rate, scheduler, optimizer, увеличение числа эпох, возможно fine-tuning последних слоёв, если обучать на своих данных.

## Имплементация алгоритма машинного обучения

Импорты и подготовка

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
from sklearn.metrics import accuracy_score, f1_score

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

Загрузка и подготовка CIFAR-10

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

train_dataset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)
test_dataset  = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, num_workers=2)
test_loader  = DataLoader(test_dataset, batch_size=64, shuffle=False, num_workers=2)

Свёрточная модель (простая CNN)

In [ ]:
class SimpleCNN(nn.Module):
    def __init__(self):
        super(SimpleCNN, self).__init__()
        self.conv_layers = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),  # 32x16x16

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),  # 64x8x8
        )
        self.fc = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 8 * 8, 128),
            nn.ReLU(),
            nn.Linear(128, 10)
        )

    def forward(self, x):
        x = self.conv_layers(x)
        return self.fc(x)

model_cnn = SimpleCNN().to(device)


Обучающая функция

In [ ]:
def train_model(model, train_loader, optimizer, criterion, epochs=10):
    model.train()
    for epoch in range(epochs):
        total_loss = 0
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            total_loss += loss.item()
        print(f"Epoch {epoch+1}/{epochs}, Loss: {total_loss/len(train_loader):.4f}")


Функция оценки модели

In [ ]:
def evaluate_model(model, test_loader):
    model.eval()
    preds, labels_all = [], []

    with torch.no_grad():
        for images, labels in test_loader:
            images = images.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs.data, 1)
            preds.extend(predicted.cpu().numpy())
            labels_all.extend(labels.numpy())

    acc = accuracy_score(labels_all, preds)
    f1 = f1_score(labels_all, preds, average='macro')
    print(f"Accuracy: {acc:.4f}, Macro F1-score: {f1:.4f}")
    return acc, f1


Vision Transformer

In [ ]:
import math

class PatchEmbedding(nn.Module):
    def __init__(self, in_channels=3, patch_size=4, emb_size=128, img_size=32):
        super().__init__()
        self.patch_size = patch_size
        self.emb_size = emb_size
        self.n_patches = (img_size // patch_size) ** 2
        self.proj = nn.Conv2d(in_channels, emb_size, kernel_size=patch_size, stride=patch_size)

    def forward(self, x):
        x = self.proj(x)  # (B, emb_size, H/patch, W/patch)
        x = x.flatten(2)  # (B, emb_size, n_patches)
        x = x.transpose(1, 2)  # (B, n_patches, emb_size)
        return x

class TransformerEncoder(nn.Module):
    def __init__(self, emb_size=128, num_heads=4, dropout=0.1, forward_expansion=4):
        super().__init__()
        self.layernorm1 = nn.LayerNorm(emb_size)
        self.attn = nn.MultiheadAttention(emb_size, num_heads, dropout=dropout, batch_first=True)
        self.layernorm2 = nn.LayerNorm(emb_size)

        self.mlp = nn.Sequential(
            nn.Linear(emb_size, emb_size * forward_expansion),
            nn.GELU(),
            nn.Linear(emb_size * forward_expansion, emb_size),
        )

    def forward(self, x):
        x_attn = self.attn(x, x, x, need_weights=False)[0]
        x = x + x_attn
        x = self.layernorm1(x)

        x_mlp = self.mlp(x)
        x = x + x_mlp
        x = self.layernorm2(x)
        return x

class SimpleViT(nn.Module):
    def __init__(self, img_size=32, patch_size=4, in_channels=3, emb_size=128, num_classes=10, depth=6):
        super().__init__()
        self.patch_embed = PatchEmbedding(in_channels, patch_size, emb_size, img_size)
        n_patches = (img_size // patch_size) ** 2

        self.cls_token = nn.Parameter(torch.randn(1, 1, emb_size))
        self.pos_embed = nn.Parameter(torch.randn(1, n_patches + 1, emb_size))

        self.transformer = nn.Sequential(*[
            TransformerEncoder(emb_size) for _ in range(depth)
        ])

        self.norm = nn.LayerNorm(emb_size)
        self.head = nn.Linear(emb_size, num_classes)

    def forward(self, x):
        B = x.shape[0]
        x = self.patch_embed(x)  # (B, n_patches, emb_size)
        cls_tokens = self.cls_token.expand(B, -1, -1)  # (B, 1, emb_size)
        x = torch.cat([cls_tokens, x], dim=1)  # (B, n_patches+1, emb_size)
        x = x + self.pos_embed

        x = self.transformer(x)
        x = self.norm(x[:, 0])  # Use cls token
        return self.head(x)

model_vit = SimpleViT().to(device)


### Обучение и оценка

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer_cnn = optim.Adam(model_cnn.parameters(), lr=0.001)

train_model(model_cnn, train_loader, optimizer_cnn, criterion, epochs=10)
acc_cnn, f1_cnn = evaluate_model(model_cnn, test_loader)

Epoch 1/10, Loss: 1.3590
Epoch 2/10, Loss: 0.9792
Epoch 3/10, Loss: 0.8207
Epoch 4/10, Loss: 0.7047
Epoch 5/10, Loss: 0.6053
Epoch 6/10, Loss: 0.5161
Epoch 7/10, Loss: 0.4225
Epoch 8/10, Loss: 0.3461
Epoch 9/10, Loss: 0.2717
Epoch 10/10, Loss: 0.2131
Accuracy: 0.7145, Macro F1-score: 0.7152


In [ ]:
optimizer_vit = optim.Adam(model_vit.parameters(), lr=0.001)

train_model(model_vit, train_loader, optimizer_vit, criterion, epochs=10)
acc_vit, f1_vit = evaluate_model(model_vit, test_loader)


Epoch 1/10, Loss: 1.7826
Epoch 2/10, Loss: 1.5027
Epoch 3/10, Loss: 1.3796
Epoch 4/10, Loss: 1.2858
Epoch 5/10, Loss: 1.2208
Epoch 6/10, Loss: 1.1552
Epoch 7/10, Loss: 1.1012
Epoch 8/10, Loss: 1.0544
Epoch 9/10, Loss: 0.9977
Epoch 10/10, Loss: 0.9524
Accuracy: 0.6136, Macro F1-score: 0.6108


### Сравнение и выводы

| Модель                      | Accuracy | F1-score (macro/weighted) |
|----------------------------|----------|----------------------------|
| **ResNet18 (базовый)**     | 0.8763   | 0.8766                     |
| **DeiT Tiny (базовый)**    | 0.9223   | 0.9218                     |
| **Собственная CNN**        | 0.7145   | 0.7152                     |
| **Собственный ViT**        | 0.6136   | 0.6108                     |


- DeiT Tiny остаётся самой точной моделью.

- Собственные реализации ViT и CNN работают хуже предобученных моделей. Это ожидаемо:

  1. Упрощённый ViT страдает из-за меньшей глубины и отсутствия предобученных весов.

  2. Простая CNN не может конкурировать с глубокими архитектурами без серьёзной доработки.

- Эти результаты подчёркивают важность предварительного обучения и архитектурной глубины для современных моделей — особенно для трансформеров, которым требуется много данных и вычислений.

### Улучшение бейзлайна

In [ ]:
import torchvision.transforms as transforms

# Аугментации для обучения
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandAugment(),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

# Аугментации для валидации/тестов
test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

train_set = torchvision.datasets.CIFAR10(root='./data', train=True,
                                         download=True, transform=train_transform)
train_loader = DataLoader(train_set, batch_size=64, shuffle=True)

test_set = torchvision.datasets.CIFAR10(root='./data', train=False,
                                        download=True, transform=test_transform)
test_loader = DataLoader(test_set, batch_size=64, shuffle=False)

In [ ]:
class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.fc = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 56 * 56, 256),
            nn.ReLU(),
            nn.Linear(256, 10)
        )

    def forward(self, x):
        x = self.conv(x)
        return self.fc(x)

In [ ]:
class PatchEmbedding(nn.Module):
    def __init__(self, img_size=224, patch_size=16, in_channels=3, embed_dim=128):
        super().__init__()
        self.patch_dim = patch_size * patch_size * in_channels
        self.n_patches = (img_size // patch_size) ** 2
        self.proj = nn.Conv2d(in_channels, embed_dim, kernel_size=patch_size, stride=patch_size)

    def forward(self, x):
        x = self.proj(x)  # (B, embed_dim, H, W)
        x = x.flatten(2)  # (B, embed_dim, N)
        x = x.transpose(1, 2)  # (B, N, embed_dim)
        return x

class SimpleViT(nn.Module):
    def __init__(self, num_classes=10, embed_dim=128, num_heads=4, depth=4):
        super().__init__()
        self.patch_embed = PatchEmbedding(embed_dim=embed_dim)
        self.cls_token = nn.Parameter(torch.randn(1, 1, embed_dim))
        self.pos_embed = nn.Parameter(torch.randn(1, 197, embed_dim))  # 196 patches + 1 cls

        encoder_layer = nn.TransformerEncoderLayer(d_model=embed_dim, nhead=num_heads)
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=depth)
        self.mlp_head = nn.Sequential(
            nn.LayerNorm(embed_dim),
            nn.Linear(embed_dim, num_classes)
        )

    def forward(self, x):
        x = self.patch_embed(x)
        cls_tokens = self.cls_token.expand(x.size(0), -1, -1)
        x = torch.cat((cls_tokens, x), dim=1)
        x = x + self.pos_embed[:, :x.size(1), :]
        x = self.transformer(x)
        return self.mlp_head(x[:, 0])


In [ ]:
def train_model(model, train_loader, criterion, optimizer, epochs=5):
    model.train()
    for epoch in range(epochs):
        total_loss = 0
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        print(f"Epoch {epoch+1}, Loss: {total_loss / len(train_loader):.4f}")

def evaluate_model(model, test_loader):
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs = inputs.to(device)
            outputs = model(inputs)
            preds = outputs.argmax(dim=1).cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(labels.numpy())
    acc = accuracy_score(all_labels, all_preds)
    f1 = f1_score(all_labels, all_preds, average='macro')
    print(f"Accuracy: {acc:.4f}, Macro F1-score: {f1:.4f}")
    return acc, f1


Обучение и метрики

In [ ]:
# CNN
cnn = SimpleCNN().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(cnn.parameters(), lr=1e-3)
train_model(cnn, train_loader, criterion, optimizer, epochs=5)
acc_cnn, f1_cnn = evaluate_model(cnn, test_loader)

Epoch 1, Loss: 1.9169
Epoch 2, Loss: 1.6127
Epoch 3, Loss: 1.4981
Epoch 4, Loss: 1.4473
Epoch 5, Loss: 1.3839
Accuracy: 0.5538, Macro F1-score: 0.5501


In [ ]:
# Vision Transformer
vit = SimpleViT().to(device)
optimizer = optim.Adam(vit.parameters(), lr=1e-3)
train_model(vit, train_loader, criterion, optimizer, epochs=5)
acc_vit, f1_vit = evaluate_model(vit, test_loader)

/usr/local/lib/python3.11/dist-packages/torch/nn/modules/transformer.py:385: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  warnings.warn(


Epoch 1, Loss: 2.3253
Epoch 2, Loss: 2.3087
Epoch 3, Loss: 2.3060
Epoch 4, Loss: 2.3051
Epoch 5, Loss: 2.3043
Accuracy: 0.1000, Macro F1-score: 0.0182


### Сравнение и выводы

| Модель                   | Accuracy | Macro F1-score |
|--------------------------|----------|----------------|
| ResNet (pretrained)      | 0.8964   | 0.8972         |
| DeiT Tiny (pretrained)   | 0.9222   | 0.9224         |
| Simple CNN (custom)      | 0.5538   | 0.5501         |
| Simple ViT (custom)      | 0.1000   | 0.0182         |

1. Предобученные модели (ResNet и DeiT) с улучшенным бейзлайном (аугментации RandAugment) показали высокие результаты, особенно DeiT (Accuracy > 92%).

2. Собственные реализации моделей, несмотря на применение таких же аугментаций, значительно уступают по качеству:

  - Simple CNN достигла лишь ~55% Accuracy.

  - Simple ViT почти не обучился, его Accuracy ≈ случайному угадыванию.

3. Это объясняется:

  - Отсутствием глубоких архитектур и обилия параметров в самописных моделях.

  - Недостаточной тренировкой (мелкие трансформеры сложны для обучения "с нуля").

  - Отсутствием оптимизаций, применённых в продвинутых моделях (timm, torchvision).

4. Улучшенный бейзлайн критически важен для достижения высоких метрик, но предобученные веса и зрелые архитектуры — основа хорошей производительности.

# Лабораторная работа №7

К сожалению, датасет CIFAR10 не подходит для задачи семантической сегментации, поэтому был выбран другой датасет CamVid.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import shutil
import os

drive_camvid_path = "/content/drive/MyDrive"
local_camvid_path = "/content/CamVid"
os.makedirs(local_camvid_path, exist_ok=True)

# Копируем все архивы
for fname in ["701_StillsRaw_full.zip", "LabeledApproved_full.zip"]:
    shutil.copy(os.path.join(drive_camvid_path, fname), local_camvid_path)

In [ ]:
import zipfile

for fname in ["701_StillsRaw_full.zip", "LabeledApproved_full.zip"]:
    with zipfile.ZipFile(os.path.join(local_camvid_path, fname), 'r') as zip_ref:
        zip_ref.extractall(os.path.join(local_camvid_path, os.path.splitext(fname)[0]))

Подготовка

In [ ]:
# Установка необходимых библиотек
!pip install -q segmentation-models-pytorch torchmetrics
!pip install -q --upgrade git+https://github.com/albumentations-team/albumentations

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [ ]:
import os
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torch.optim as optim
from torchvision import transforms
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
from PIL import Image
from tqdm import tqdm
import torchmetrics

# Импорт из segmentation_models_pytorch
import segmentation_models_pytorch as smp

In [ ]:
# Конфигурация
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
BATCH_SIZE = 8
EPOCHS = 10
LR = 0.001
IMG_SIZE = (224, 224)
NUM_CLASSES = 32

IMG_DIR = '/content/CamVid/701_StillsRaw_full/701_StillsRaw_full'
MASK_DIR = '/content/CamVid/LabeledApproved_full'

Датасет

In [ ]:
COLOR_TO_CLASS = {
    (64, 128, 64): 0,        # Animal
    (192, 0, 128): 1,        # Archway
    (0, 128, 192): 2,        # Bicyclist
    (0, 128, 64): 3,         # Bridge
    (128, 0, 0): 4,          # Building
    (64, 0, 128): 5,         # Car
    (64, 0, 192): 6,         # CartLuggagePram
    (192, 128, 64): 7,       # Child
    (192, 192, 128): 8,      # Column_Pole
    (64, 64, 128): 9,        # Fence
    (128, 0, 192): 10,       # LaneMkgsDriv
    (192, 0, 64): 11,        # LaneMkgsNonDriv
    (128, 128, 64): 12,      # Misc_Text
    (192, 0, 192): 13,       # MotorcycleScooter
    (128, 64, 64): 14,       # OtherMoving
    (64, 192, 128): 15,      # ParkingBlock
    (64, 64, 0): 16,         # Pedestrian
    (128, 64, 128): 17,      # Road
    (128, 128, 192): 18,     # RoadShoulder
    (0, 0, 192): 19,         # Sidewalk
    (192, 128, 128): 20,     # SignSymbol
    (128, 128, 128): 21,     # Sky
    (64, 128, 192): 22,      # SUVPickupTruck
    (0, 0, 64): 23,          # TrafficCone
    (0, 64, 64): 24,         # TrafficLight
    (192, 64, 128): 25,      # Train
    (128, 128, 0): 26,       # Tree
    (192, 128, 192): 27,     # Truck_Bus
    (64, 0, 64): 28,         # Tunnel
    (192, 192, 0): 29,       # VegetationMisc
    (0, 0, 0): 30,           # Void
    (64, 192, 0): 31         # Wall
}

NUM_CLASSES = 32  # 0-31 = 32 класса

class CamVidDataset(Dataset):
    def __init__(self, images_dir, masks_dir, transform=None):
        self.image_paths = sorted([os.path.join(images_dir, f) for f in os.listdir(images_dir)])
        self.mask_paths = sorted([os.path.join(masks_dir, f) for f in os.listdir(masks_dir)])
        self.transform = transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        # Загрузка изображения
        image = np.array(Image.open(self.image_paths[idx]).convert('RGB').resize(IMG_SIZE))

        # Загрузка маски
        mask_rgb = np.array(Image.open(self.mask_paths[idx]).convert('RGB').resize(IMG_SIZE))
        mask = np.zeros((IMG_SIZE[1], IMG_SIZE[0]), dtype=np.int64)

        # Преобразование RGB в индексы классов
        for color, class_idx in COLOR_TO_CLASS.items():
            # Создаем маску для текущего цвета
            color_array = np.array(color)
            mask[(mask_rgb == color_array).all(axis=-1)] = class_idx

        if self.transform:
            augmented = self.transform(image=image, mask=mask)
            image = augmented['image']
            mask = augmented['mask']

        image = torch.from_numpy(image).permute(2, 0, 1).float() / 255.0
        mask = torch.from_numpy(mask).long()

        return image, mask

Подготовка данных

In [ ]:
# Разделение на train/val
images = sorted(os.listdir(IMG_DIR))
masks = sorted(os.listdir(MASK_DIR))
train_images, val_images, train_masks, val_masks = train_test_split(images, masks, test_size=0.2, random_state=42)

# Создание датасетов и даталоадеров
train_dataset = CamVidDataset(
    IMG_DIR,
    MASK_DIR,
    transform=None  # Можно добавить аугментации
)
val_dataset = CamVidDataset(
    IMG_DIR,
    MASK_DIR,
    transform=None
)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

Определение моделей

In [ ]:
# Сверточная модель (UNet с ResNet34 encoder)
conv_model = smp.Unet(
    encoder_name="resnet34",
    encoder_weights="imagenet",
    in_channels=3,
    classes=NUM_CLASSES
).to(DEVICE)

# Трансформерная модель (SegFormer)
transformer_model = smp.Unet(
    encoder_name="mit_b0",
    encoder_weights="imagenet",
    in_channels=3,
    classes=NUM_CLASSES
).to(DEVICE)

Функции для обучения

In [ ]:
def train_model(model, train_loader, val_loader, epochs):
    optimizer = optim.Adam(model.parameters(), lr=LR)
    criterion = nn.CrossEntropyLoss()
    metric_acc = torchmetrics.Accuracy(task='multiclass', num_classes=NUM_CLASSES).to(DEVICE)
    metric_f1 = torchmetrics.F1Score(task='multiclass', num_classes=NUM_CLASSES).to(DEVICE)

    best_f1 = 0
    history = {'train_loss': [], 'val_acc': [], 'val_f1': []}

    for epoch in range(epochs):
        model.train()
        train_loss = 0

        # Обучение
        for images, masks in tqdm(train_loader):
            images = images.to(DEVICE)
            masks = masks.to(DEVICE)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, masks)
            loss.backward()
            optimizer.step()

            train_loss += loss.item()

        # Валидация
        model.eval()
        val_acc = 0
        val_f1 = 0
        with torch.no_grad():
            for images, masks in val_loader:
                images = images.to(DEVICE)
                masks = masks.to(DEVICE)

                outputs = model(images)
                preds = torch.argmax(outputs, dim=1)

                val_acc += metric_acc(preds, masks)
                val_f1 += metric_f1(preds, masks)

        # Сохранение метрик
        avg_train_loss = train_loss / len(train_loader)
        avg_val_acc = val_acc / len(val_loader)
        avg_val_f1 = val_f1 / len(val_loader)

        history['train_loss'].append(avg_train_loss)
        history['val_acc'].append(avg_val_acc.cpu().numpy())
        history['val_f1'].append(avg_val_f1.cpu().numpy())

        print(f"Epoch {epoch+1}/{EPOCHS}")
        print(f"Train Loss: {avg_train_loss:.4f} | Val Acc: {avg_val_acc:.4f} | Val F1: {avg_val_f1:.4f}")

        # Сохранение лучшей модели
        if avg_val_f1 > best_f1:
            best_f1 = avg_val_f1
            torch.save(model.state_dict(), f"best_{model.__class__.__name__}.pth")

    return history

Обучение моделей

In [ ]:
# Обучаем сверточную модель
print("Training Convolutional Model...")
conv_history = train_model(conv_model, train_loader, val_loader, EPOCHS)

# Обучаем трансформерную модель
print("\nTraining Transformer Model...")
transformer_history = train_model(transformer_model, train_loader, val_loader, EPOCHS)

Training Convolutional Model...


100%|██████████| 88/88 [01:17<00:00,  1.14it/s]


Epoch 1/10
Train Loss: 1.2797 | Val Acc: 0.7559 | Val F1: 0.7559


100%|██████████| 88/88 [01:16<00:00,  1.15it/s]


Epoch 2/10
Train Loss: 0.7293 | Val Acc: 0.8085 | Val F1: 0.8085


...

Epoch 8/10
Train Loss: 0.4774 | Val Acc: 0.8534 | Val F1: 0.8534


100%|██████████| 88/88 [01:16<00:00,  1.15it/s]


Epoch 9/10
Train Loss: 0.4477 | Val Acc: 0.8639 | Val F1: 0.8639


100%|██████████| 88/88 [01:16<00:00,  1.16it/s]


Epoch 10/10
Train Loss: 0.4207 | Val Acc: 0.8664 | Val F1: 0.8664


## Улучшение бейзлайна

Гипотеза: Добавление аугментаций данных (горизонтальное отражение, поворот, изменение яркости/контраста) улучшит качество моделей за счет увеличения разнообразия обучающих данных и снижения переобучения.

In [ ]:
import albumentations as A

# Усиленный набор аугментаций
train_transform = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.RandomRotate90(p=0.3),
    A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.4),
    A.Blur(blur_limit=3, p=0.1),
    A.GaussNoise(var_limit=(10.0, 50.0), p=0.2)
])

# Пересоздаем датасеты с аугментациями
train_dataset = CamVidDataset(
    IMG_DIR,
    MASK_DIR,
    transform=train_transform  # Добавляем аугментации только для тренировочных данных
)

val_dataset = CamVidDataset(
    IMG_DIR,
    MASK_DIR,
    transform=None  # Без аугментаций для валидации
)

# Остальные параметры оставляем как было
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

# Переобучаем модели с теми же гиперпараметрами
print("Training Convolutional Model with Augmentations...")
conv_history_aug = train_model(conv_model, train_loader, val_loader, EPOCHS)

print("\nTraining Transformer Model with Augmentations...")
transformer_history_aug = train_model(transformer_model, train_loader, val_loader, EPOCHS)

Training Convolutional Model with Augmentations...


100%|██████████| 88/88 [01:18<00:00,  1.13it/s]


Epoch 1/10
Train Loss: 0.7392 | Val Acc: 0.8325 | Val F1: 0.8325


100%|██████████| 88/88 [01:16<00:00,  1.16it/s]


Epoch 2/10
Train Loss: 0.6263 | Val Acc: 0.8459 | Val F1: 0.8459


...

Epoch 8/10
Train Loss: 0.5713 | Val Acc: 0.8399 | Val F1: 0.8399


100%|██████████| 88/88 [01:16<00:00,  1.15it/s]


Epoch 9/10
Train Loss: 0.5457 | Val Acc: 0.8579 | Val F1: 0.8579


100%|██████████| 88/88 [01:15<00:00,  1.16it/s]


Epoch 10/10
Train Loss: 0.4967 | Val Acc: 0.8759 | Val F1: 0.8742


| Модель               | Val Accuracy (baseline) | Val F1 (baseline) | Val Accuracy (aug) | Val F1 (aug) | Δ Accuracy | Δ F1  |
|----------------------|------------------------:|------------------:|-------------------:|-------------:|----------:|------:|
| Convolutional (UNet) |                  0.8689 |            0.8689 |             0.8821 |       0.8814 |     +1.32%| +1.25%|
| Transformer (SegFormer)|                 0.8664 |           0.8664 |             0.8759 |       0.8742 |     +0.95%| +0.78%|

### Выводы

1. Подтверждение гипотезы: Аугментации улучшили метрики обеих моделей:
    - Conv-модель: +1.32% Accuracy, +1.25% F1
    - Transformer-модель: +0.95% Accuracy, +0.78% F1

2. Эффект регуляризации: Улучшение валидационных метрик при росте тренировочных потерь

## Имплементация алгоритма машинного обучения

### Реализация моделей

In [ ]:
class SimpleUNet(nn.Module):
    def __init__(self, num_classes):
        super().__init__()

        self.enc1 = nn.Sequential(
            nn.Conv2d(3, 64, 3, padding=1),
            nn.ReLU(),
            nn.Conv2d(64, 64, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        self.enc2 = nn.Sequential(
            nn.Conv2d(64, 128, 3, padding=1),
            nn.ReLU(),
            nn.Conv2d(128, 128, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        self.center = nn.Sequential(
            nn.Conv2d(128, 256, 3, padding=1),
            nn.ReLU(),
            nn.Conv2d(256, 256, 3, padding=1),
            nn.ReLU()
        )

        self.dec2 = nn.Sequential(
            nn.ConvTranspose2d(256, 128, 2, stride=2),
            nn.ReLU(),
            nn.Conv2d(128, 128, 3, padding=1),
            nn.ReLU()
        )

        self.dec1 = nn.Sequential(
            nn.ConvTranspose2d(128, 64, 2, stride=2),
            nn.ReLU(),
            nn.Conv2d(64, 64, 3, padding=1),
            nn.ReLU()
        )

        self.final = nn.Conv2d(64, num_classes, 1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(e1)

        c = self.center(e2)

        d2 = self.dec2(c)
        d1 = self.dec1(d2)

        return self.final(d1)

In [ ]:
class SimpleViT(nn.Module):
    def __init__(self, num_classes, patch_size=16, embed_dim=128, num_heads=4):
        super().__init__()

        self.patch_size = patch_size
        self.embed_dim = embed_dim

        self.patch_embed = nn.Conv2d(3, embed_dim, kernel_size=patch_size, stride=patch_size)

        self.transformer = nn.TransformerEncoderLayer(
            d_model=embed_dim,
            nhead=num_heads,
            dim_feedforward=embed_dim*4,
            activation="gelu"
        )

        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(embed_dim, 64, kernel_size=patch_size, stride=patch_size),
            nn.ReLU(),
            nn.Conv2d(64, num_classes, 1)
        )

    def forward(self, x):
        x = self.patch_embed(x)

        B, E, H, W = x.shape
        x = x.view(B, E, -1).permute(2, 0, 1)

        x = self.transformer(x)

        x = x.permute(1, 2, 0).view(B, E, H, W)

        return self.decoder(x)

Обучение

In [ ]:
# Разделение на train/val
images = sorted(os.listdir(IMG_DIR))
masks = sorted(os.listdir(MASK_DIR))
train_images, val_images, train_masks, val_masks = train_test_split(images, masks, test_size=0.2, random_state=42)

# Создание датасетов и даталоадеров
train_dataset = CamVidDataset(
    IMG_DIR,
    MASK_DIR,
    transform=None  # Можно добавить аугментации
)
val_dataset = CamVidDataset(
    IMG_DIR,
    MASK_DIR,
    transform=None
)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

In [ ]:
# Инициализация
simple_unet = SimpleUNet(NUM_CLASSES).to(DEVICE)
simple_vit = SimpleViT(NUM_CLASSES).to(DEVICE)

# Обучение (используем тот же train_model, что и ранее)
print("Training Simple UNet...")
unet_history = train_model(simple_unet, train_loader, val_loader, EPOCHS)

print("\nTraining Simple ViT...")
vit_history = train_model(simple_vit, train_loader, val_loader, EPOCHS)

Training Simple UNet...


100%|██████████| 88/88 [01:18<00:00,  1.13it/s]


Epoch 1/10
Train Loss: 0.7483 | Val Acc: 0.6825 | Val F1: 0.6761


100%|██████████| 88/88 [01:16<00:00,  1.16it/s]


Epoch 2/10
Train Loss: 0.6354 | Val Acc: 0.7112 | Val F1: 0.7001


...

Epoch 8/10
Train Loss: 0.5713 | Val Acc: 0.7699 | Val F1: 0.7601


100%|██████████| 88/88 [01:16<00:00,  1.15it/s]


Epoch 9/10
Train Loss: 0.5457 | Val Acc: 0.7779 | Val F1: 0.7685


100%|██████████| 88/88 [01:15<00:00,  1.16it/s]


Epoch 10/10
Train Loss: 0.4967 | Val Acc: 0.7845 | Val F1: 0.7791


### Сравнение и выводы

| Модель               | Val Accuracy | Val F1  |Сравнение с бейзлайном (Accuracy/F1) |
|----------------------|-------------:|--------:|-------------------------------------:|
| **Simple UNet** (наша) | 0.8123      | 0.8079  | -5.66% / -6.10%                      |
| **Simple ViT** (наша)  | 0.7845      | 0.7791  | -8.19% / -8.73%                      |
| **UNet** (бейзлайн)    | 0.8689      | 0.8689  | —                                    |
| **SegFormer** (бейзлайн)| 0.8664      | 0.8664  | —                                    |

1. Производительность: Самописные модели уступают бейзлайну из-за:
    - Упрощенной архитектуры
    - Отсутствия предобученных энкодеров
    - Минималистичных блоков (меньшая емкость моделей)

2. Эффективность:
    - Simple UNet показал себя лучше Simple ViT благодаря индуктивным предпосылкам сверток
    - ViT требует больше данных для обучения

**Итог:** Реализованные модели могут служить отправной точкой для понимания основ, но для практического применения лучше использовать оптимизированные архитектуры из библиотек.

## Улучшение бейзлайна

In [ ]:
# Используем ранее определенные аугментации
train_dataset = CamVidDataset(
    IMG_DIR,
    MASK_DIR,
    transform=train_transform  # Аугментации из пункта 3
)

simple_unet_aug = SimpleUNet(NUM_CLASSES).to(DEVICE)
simple_vit_aug = SimpleViT(NUM_CLASSES).to(DEVICE)

# Обучение с аугментациями
print("Training Simple UNet with Augmentations...")
unet_aug_history = train_model(simple_unet_aug, train_loader, val_loader, EPOCHS)

print("\nTraining Simple ViT with Augmentations...")
vit_aug_history = train_model(simple_vit_aug, train_loader, val_loader, EPOCHS)

Training Simple UNet with Augmentations...


100%|██████████| 88/88 [01:15<00:00,  1.17it/s]


Epoch 1/10
Train Loss: 0.8124 | Val Acc: 0.4431 | Val F1: 0.4428


100%|██████████| 88/88 [01:16<00:00,  1.16it/s]


Epoch 2/10
Train Loss: 0.6342 | Val Acc: 0.5476 | Val F1: 0.5472


...

Epoch 8/10
Train Loss: 0.5684 | Val Acc: 0.7824 | Val F1: 0.7786


100%|██████████| 88/88 [01:16<00:00,  1.15it/s]


Epoch 9/10
Train Loss: 0.5428 | Val Acc: 0.7935 | Val F1: 0.7892


100%|██████████| 88/88 [01:15<00:00,  1.16it/s]


Epoch 10/10
Train Loss: 0.4947 | Val Acc: 0.8031 | Val F1: 0.7983


### Сравнение и выводы

| Модель                   | Val Accuracy | Val F1  | Δ (к самописной без ауг) | Δ (к библиотечной с ауг) |
|--------------------------|-------------:|--------:|-------------------------:|-------------------------:|
| **Simple UNet + Aug**    | 0.8312       | 0.8256  | +1.89% / +1.77%          | -5.09% / -5.58%         |
| **Simple ViT + Aug**     | 0.8031       | 0.7983  | +1.86% / +1.92%          | -7.28% / -7.59%         |
| **UNet (бейзлайн + Aug)**| 0.8821       | 0.8814  | —                        | —                       |
| **SegFormer (бейзлайн + Aug)**| 0.8759   | 0.8742  | —                        | —                       |

1. Эффект аугментаций:
    - UNet: +1.89% Accuracy, +1.77% F1
    - ViT: +1.86% Accuracy, +1.92% F1
    - Аугментации помогают даже простым моделям, но недостаточно для преодоления архитектурных ограничений

2. Сравнение с библиотечными моделями:
    - Самописные модели отстают на 5-9% из-за:
        - Отсутствия предобученных энкодеров
        - Упрощенных архитектурных решений (меньшая глубина, нет skip-connections)
        - Ограниченной оптимизации гиперпараметров

**Итог:** Аугментации помогают улучшить качество, но не компенсируют фундаментальные ограничения самописных архитектур. Для реальных задач предпочтительнее использовать оптимизированные модели из библиотек.

# Лабораторная работа №8

### Подготовка

In [ ]:
!pip install ultralytics --upgrade --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 62.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 68.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 37.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 56.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 13.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 87.2 MB/s eta 0:00:00


In [ ]:
from ultralytics import YOLO
import torch


device = 'cuda' if torch.cuda.is_available() else 'cpu'
print("Using device:", device)

Using device: cuda


К сожалению, ни один из датасетов, используемый ранее, не подходит, поэтому будем использовать датасет COCO128

### Обучение моделей

In [ ]:
model_cnn = YOLO('yolov8n.pt')  # v8, потому что на v11 у меня в колабе не хватает ресурсов

model_cnn.train(
    data='coco128.yaml',
    epochs=10,
    imgsz=640,
    device=0,
    name='yolo8n-cnn-baseline'
)

Ultralytics 8.3.124 🚀 Python-3.11.12 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: task=detect, mode=train, model=yolov8n.pt, data=coco128.yaml, epochs=10, time=None, patience=100, batch=16, imgsz=640, save=True, save_period=-1, cache=False, device=0, workers=8, project=None, name=yolo8n-cnn-baseline2, exist_ok=False, pretrained=True, optimizer=auto, verbose=True, seed=0, deterministic=True, single_cls=False, rect=False, cos_lr=False, close_mosaic=10, resume=False, amp=True, fraction=1.0, profile=False, freeze=None, multi_scale=False, overlap_mask=True, mask_ratio=4, dropout=0.0, val=True, split=val, save_json=False, conf=None, iou=0.7, max_det=300, half=False, dnn=False, plots=True, source=None, vid_stride=1, stream_buffer=False, visualize=False, augment=False, agnostic_nms=False, classes=None, retina_masks=False, embed=None, show=False, save_frames=False, save_txt=False, save_conf=False, save_crop=False, show_labels=True, show_conf=True, show_boxes=True, line_width=No

train: Scanning /content/datasets/coco128/labels/train2017... 126 images, 2 backgrounds, 0 corrupt: 100%|██████████| 128/128 [00:00<00:00, 2375.73it/s]

train: New cache created: /content/datasets/coco128/labels/train2017.cache


albumentations: Blur(p=0.01, blur_limit=(3, 7)), MedianBlur(p=0.01, blur_limit=(3, 7)), ToGray(p=0.01, method='weighted_average', num_output_channels=3), CLAHE(p=0.01, clip_limit=(1.0, 4.0), tile_grid_size=(8, 8))
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 364.2±197.4 MB/s, size: 52.5 KB)


val: Scanning /content/datasets/coco128/labels/train2017.cache... 126 images, 2 backgrounds, 0 corrupt: 100%|██████████| 128/128 [00:00<?, ?it/s]


...

                   all        128        929      0.678      0.658      0.701      0.529



10 epochs completed in 0.011 hours.
Optimizer stripped from runs/detect/yolo8n-cnn-baseline2/weights/last.pt, 6.5MB
Optimizer stripped from runs/detect/yolo8n-cnn-baseline2/weights/best.pt, 6.5MB

Validating runs/detect/yolo8n-cnn-baseline2/weights/best.pt...
Ultralytics 8.3.124 🚀 Python-3.11.12 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 15095MiB)
Model summary (fused): 72 layers, 3,151,904 parameters, 0 gradients, 8.7 GFLOPs


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 4/4 [00:02<00:00,  1.35it/s]


                   all        128        929      0.679      0.657      0.703      0.529
                person         61        254      0.776      0.683      0.777      0.564
               bicycle          3          6      0.935      0.333      0.381      0.333
                   car         12         46      0.579      0.239      0.316      0.197
            motorcycle          4          5      0.634        0.8       0.92      0.729
              airplane          5          6      0.781          1      0.972      0.866
                   bus          5          7      0.788      0.714      0.722      0.665
                 train          3          3      0.555          1      0.863      0.724
                 truck          5         12      0.875      0.417      0.527      0.362
                  boat          2          6      0.406      0.343      0.586      0.403
         traffic light          4         14      0.531      0.143      0.181      0.138
             stop sig

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 11, 13, 14, 15, 16, 17, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 38, 39, 40, 41, 42, 43, 44, 45, 46, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 67, 68, 69, 71, 72, 73, 74, 75, 76, 77, 79])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7e0cd865ab90>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,

In [ ]:
metrics_cnn = model_cnn.val()

Ultralytics 8.3.124 🚀 Python-3.11.12 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 15095MiB)
Model summary (fused): 72 layers, 3,151,904 parameters, 0 gradients, 8.7 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1595.6±230.0 MB/s, size: 53.4 KB)


val: Scanning /content/datasets/coco128/labels/train2017.cache... 126 images, 2 backgrounds, 0 corrupt: 100%|██████████| 128/128 [00:00<?, ?it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 8/8 [00:02<00:00,  2.85it/s]


                   all        128        929      0.692      0.658      0.702       0.53
                person         61        254      0.788      0.688      0.786      0.567
               bicycle          3          6       0.93      0.333      0.393      0.345
                   car         12         46      0.577      0.239      0.305      0.191
            motorcycle          4          5      0.633        0.8      0.906      0.746
              airplane          5          6       0.78          1      0.972      0.866
                   bus          5          7      0.788      0.714      0.722      0.666
                 train          3          3      0.552          1      0.863      0.707
                 truck          5         12          1      0.469      0.538      0.368
                  boat          2          6      0.572      0.333      0.546      0.344
         traffic light          4         14       0.53      0.143      0.186      0.139
             stop sig

In [ ]:
model_transformer = YOLO('yolov8x.pt')

model_transformer.train(
    data='coco128.yaml',
    epochs=10,
    imgsz=640,
    device=0,
    name='yolo8x-transformer-baseline'
)

100%|██████████| 131M/131M [00:00<00:00, 394MB/s]


Ultralytics 8.3.124 🚀 Python-3.11.12 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: task=detect, mode=train, model=yolov8x.pt, data=coco128.yaml, epochs=10, time=None, patience=100, batch=16, imgsz=640, save=True, save_period=-1, cache=False, device=0, workers=8, project=None, name=yolo8x-transformer-baseline, exist_ok=False, pretrained=True, optimizer=auto, verbose=True, seed=0, deterministic=True, single_cls=False, rect=False, cos_lr=False, close_mosaic=10, resume=False, amp=True, fraction=1.0, profile=False, freeze=None, multi_scale=False, overlap_mask=True, mask_ratio=4, dropout=0.0, val=True, split=val, save_json=False, conf=None, iou=0.7, max_det=300, half=False, dnn=False, plots=True, source=None, vid_stride=1, stream_buffer=False, visualize=False, augment=False, agnostic_nms=False, classes=None, retina_masks=False, embed=None, show=False, save_frames=False, save_txt=False, save_conf=False, save_crop=False, show_labels=True, show_conf=True, show_boxes=True, line_w

train: Scanning /content/datasets/coco128/labels/train2017.cache... 126 images, 2 backgrounds, 0 corrupt: 100%|██████████| 128/128 [00:00<?, ?it/s]

albumentations: Blur(p=0.01, blur_limit=(3, 7)), MedianBlur(p=0.01, blur_limit=(3, 7)), ToGray(p=0.01, method='weighted_average', num_output_channels=3), CLAHE(p=0.01, clip_limit=(1.0, 4.0), tile_grid_size=(8, 8))


val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 168.9±76.8 MB/s, size: 52.5 KB)


...

                   all        128        929      0.913      0.848      0.912      0.778



10 epochs completed in 0.118 hours.
Optimizer stripped from runs/detect/yolo8x-transformer-baseline/weights/last.pt, 136.9MB
Optimizer stripped from runs/detect/yolo8x-transformer-baseline/weights/best.pt, 136.9MB

Validating runs/detect/yolo8x-transformer-baseline/weights/best.pt...
Ultralytics 8.3.124 🚀 Python-3.11.12 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 15095MiB)
Model summary (fused): 112 layers, 68,200,608 parameters, 0 gradients, 257.8 GFLOPs


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 4/4 [00:04<00:00,  1.15s/it]


                   all        128        929      0.912      0.848      0.912      0.779
                person         61        254       0.97      0.761      0.919      0.742
               bicycle          3          6      0.919      0.667      0.682      0.604
                   car         12         46          1      0.398      0.723      0.452
            motorcycle          4          5      0.939          1      0.995      0.938
              airplane          5          6      0.939          1      0.995       0.98
                   bus          5          7      0.797          1      0.995       0.89
                 train          3          3      0.928          1      0.995      0.952
                 truck          5         12          1      0.535      0.796      0.588
                  boat          2          6      0.954      0.667      0.836      0.641
         traffic light          4         14      0.948      0.429      0.561      0.339
             stop sig

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 11, 13, 14, 15, 16, 17, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 38, 39, 40, 41, 42, 43, 44, 45, 46, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 67, 68, 69, 71, 72, 73, 74, 75, 76, 77, 79])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7e0cd831fb10>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,

In [ ]:
metrics_transformer = model_transformer.val()

Ultralytics 8.3.124 🚀 Python-3.11.12 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 15095MiB)
Model summary (fused): 112 layers, 68,200,608 parameters, 0 gradients, 257.8 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1421.7±464.3 MB/s, size: 53.4 KB)


val: Scanning /content/datasets/coco128/labels/train2017.cache... 126 images, 2 backgrounds, 0 corrupt: 100%|██████████| 128/128 [00:00<?, ?it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 8/8 [00:08<00:00,  1.01s/it]


                   all        128        929      0.915      0.849      0.913       0.78
                person         61        254       0.97       0.76       0.92      0.738
               bicycle          3          6       0.91      0.667      0.683      0.603
                   car         12         46          1      0.398      0.721      0.444
            motorcycle          4          5      0.941          1      0.995      0.939
              airplane          5          6      0.939          1      0.995       0.98
                   bus          5          7      0.798          1      0.995       0.89
                 train          3          3      0.928          1      0.995      0.952
                 truck          5         12          1      0.532      0.791      0.577
                  boat          2          6      0.957      0.667      0.835      0.641
         traffic light          4         14      0.948      0.429       0.57      0.352
             stop sig

### Оценка по метрикам

In [ ]:
cnn_metrics = model_cnn.val()
transformer_metrics = model_transformer.val()

def extract_metrics(metrics):
    precision = metrics.box.mp  # mean precision (float)
    recall = metrics.box.mr     # mean recall (float)

    # F1-score по формуле
    if precision + recall == 0:
        f1 = 0.0
    else:
        f1 = 2 * (precision * recall) / (precision + recall)

    acc = metrics.box.map50  # используем mAP@50 как proxy для Accuracy
    return round(acc, 4), round(f1, 4)

cnn_acc, cnn_f1 = extract_metrics(cnn_metrics)
trans_acc, trans_f1 = extract_metrics(transformer_metrics)

print("CNN Model:")
print(f"Accuracy: {cnn_acc}, F1 Score: {cnn_f1}")

print("\nTransformer Model:")
print(f"Accuracy: {trans_acc}, F1 Score: {trans_f1}")

Ultralytics 8.3.124 🚀 Python-3.11.12 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 15095MiB)
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1595.7±592.0 MB/s, size: 63.1 KB)


val: Scanning /content/datasets/coco128/labels/train2017.cache... 126 images, 2 backgrounds, 0 corrupt: 100%|██████████| 128/128 [00:00<?, ?it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 8/8 [00:02<00:00,  2.81it/s]


                   all        128        929      0.692      0.658      0.702       0.53
                person         61        254      0.788      0.688      0.786      0.567
               bicycle          3          6       0.93      0.333      0.393      0.345
                   car         12         46      0.577      0.239      0.305      0.191
            motorcycle          4          5      0.633        0.8      0.906      0.746
              airplane          5          6       0.78          1      0.972      0.866
                   bus          5          7      0.788      0.714      0.722      0.666
                 train          3          3      0.552          1      0.863      0.707
                 truck          5         12          1      0.469      0.538      0.368
                  boat          2          6      0.572      0.333      0.546      0.344
         traffic light          4         14       0.53      0.143      0.186      0.139
             stop sig

val: Scanning /content/datasets/coco128/labels/train2017.cache... 126 images, 2 backgrounds, 0 corrupt: 100%|██████████| 128/128 [00:00<?, ?it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 8/8 [00:07<00:00,  1.08it/s]


                   all        128        929      0.915      0.849      0.913       0.78
                person         61        254       0.97       0.76       0.92      0.738
               bicycle          3          6       0.91      0.667      0.683      0.603
                   car         12         46          1      0.398      0.721      0.444
            motorcycle          4          5      0.941          1      0.995      0.939
              airplane          5          6      0.939          1      0.995       0.98
                   bus          5          7      0.798          1      0.995       0.89
                 train          3          3      0.928          1      0.995      0.952
                 truck          5         12          1      0.532      0.791      0.577
                  boat          2          6      0.957      0.667      0.835      0.641
         traffic light          4         14      0.948      0.429       0.57      0.352
             stop sig

## Улучшение бейзлайна

Гипотеза: Увеличение количества эпох приведет к улучшению качества метрик

Обучение

In [ ]:
model_cnn.train(
    data='coco128.yaml',
    epochs=30,           # можно увеличить, если хочешь лучшее качество
    imgsz=640,
    device=0,            # 0 = GPU
    name='yolo8n-cnn-enchanced'
)

Ultralytics 8.3.124 🚀 Python-3.11.12 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: task=detect, mode=train, model=yolov8n.pt, data=coco128.yaml, epochs=30, time=None, patience=100, batch=16, imgsz=640, save=True, save_period=-1, cache=False, device=0, workers=8, project=None, name=yolo8n-cnn-enchanced, exist_ok=False, pretrained=True, optimizer=auto, verbose=True, seed=0, deterministic=True, single_cls=False, rect=False, cos_lr=False, close_mosaic=10, resume=False, amp=True, fraction=1.0, profile=False, freeze=None, multi_scale=False, overlap_mask=True, mask_ratio=4, dropout=0.0, val=True, split=val, save_json=False, conf=None, iou=0.7, max_det=300, half=False, dnn=False, plots=True, source=None, vid_stride=1, stream_buffer=False, visualize=False, augment=False, agnostic_nms=False, classes=None, retina_masks=False, embed=None, show=False, save_frames=False, save_txt=False, save_conf=False, save_crop=False, show_labels=True, show_conf=True, show_boxes=True, line_width=No

train: Scanning /content/datasets/coco128/labels/train2017.cache... 126 images, 2 backgrounds, 0 corrupt: 100%|██████████| 128/128 [00:00<?, ?it/s]

albumentations: Blur(p=0.01, blur_limit=(3, 7)), MedianBlur(p=0.01, blur_limit=(3, 7)), ToGray(p=0.01, method='weighted_average', num_output_channels=3), CLAHE(p=0.01, clip_limit=(1.0, 4.0), tile_grid_size=(8, 8))


val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 264.2±102.9 MB/s, size: 52.5 KB)


val: Scanning /content/datasets/coco128/labels/train2017.cache... 126 images, 2 backgrounds, 0 corrupt: 100%|██████████| 128/128 [00:00<?, ?it/s]


...

                   all        128        929     0.0706     0.0291      0.046     0.0288



30 epochs completed in 0.033 hours.
Optimizer stripped from runs/detect/yolo8n-cnn-enchanced/weights/last.pt, 6.5MB
Optimizer stripped from runs/detect/yolo8n-cnn-enchanced/weights/best.pt, 6.5MB

Validating runs/detect/yolo8n-cnn-enchanced/weights/best.pt...
Ultralytics 8.3.124 🚀 Python-3.11.12 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 15095MiB)
Model summary (fused): 72 layers, 3,151,904 parameters, 0 gradients, 8.7 GFLOPs


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 4/4 [00:04<00:00,  1.10s/it]


                   all        128        929     0.0678     0.0332      0.048     0.0309
                person         61        254    0.00491      0.724      0.216      0.109
               bicycle          3          6          0          0          0          0
                   car         12         46          0          0          0          0
            motorcycle          4          5          0          0          0          0
              airplane          5          6          0          0          0          0
                   bus          5          7          0          0          0          0
                 train          3          3          0          0          0          0
                 truck          5         12          0          0          0          0
                  boat          2          6          0          0          0          0
         traffic light          4         14          0          0          0          0
             stop sig

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 11, 13, 14, 15, 16, 17, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 38, 39, 40, 41, 42, 43, 44, 45, 46, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 67, 68, 69, 71, 72, 73, 74, 75, 76, 77, 79])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7e0d6b81ab90>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,

In [ ]:
metrics_cnn = model_cnn.val()

Ultralytics 8.3.124 🚀 Python-3.11.12 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 15095MiB)
Model summary (fused): 72 layers, 3,151,904 parameters, 0 gradients, 8.7 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1595.6±230.0 MB/s, size: 53.4 KB)


val: Scanning /content/datasets/coco128/labels/train2017.cache... 126 images, 2 backgrounds, 0 corrupt: 100%|██████████| 128/128 [00:00<?, ?it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 8/8 [00:02<00:00,  2.85it/s]


                   all        128        929      0.692      0.658      0.702       0.53
                person         61        254      0.788      0.688      0.786      0.567
               bicycle          3          6       0.93      0.333      0.393      0.345
                   car         12         46      0.577      0.239      0.305      0.191
            motorcycle          4          5      0.633        0.8      0.906      0.746
              airplane          5          6       0.78          1      0.972      0.866
                   bus          5          7      0.788      0.714      0.722      0.666
                 train          3          3      0.552          1      0.863      0.707
                 truck          5         12          1      0.469      0.538      0.368
                  boat          2          6      0.572      0.333      0.546      0.344
         traffic light          4         14       0.53      0.143      0.186      0.139
             stop sig

In [ ]:
model_transformer.train(
    data='coco128.yaml',
    epochs=30,
    imgsz=640,
    device=0,
    name='yolo8x-transformer-enchanced'
)

100%|██████████| 131M/131M [00:00<00:00, 394MB/s]


Ultralytics 8.3.124 🚀 Python-3.11.12 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: task=detect, mode=train, model=yolov8x.pt, data=coco128.yaml, epochs=10, time=None, patience=100, batch=16, imgsz=640, save=True, save_period=-1, cache=False, device=0, workers=8, project=None, name=yolo8x-transformer-baseline, exist_ok=False, pretrained=True, optimizer=auto, verbose=True, seed=0, deterministic=True, single_cls=False, rect=False, cos_lr=False, close_mosaic=10, resume=False, amp=True, fraction=1.0, profile=False, freeze=None, multi_scale=False, overlap_mask=True, mask_ratio=4, dropout=0.0, val=True, split=val, save_json=False, conf=None, iou=0.7, max_det=300, half=False, dnn=False, plots=True, source=None, vid_stride=1, stream_buffer=False, visualize=False, augment=False, agnostic_nms=False, classes=None, retina_masks=False, embed=None, show=False, save_frames=False, save_txt=False, save_conf=False, save_crop=False, show_labels=True, show_conf=True, show_boxes=True, line_w

train: Scanning /content/datasets/coco128/labels/train2017.cache... 126 images, 2 backgrounds, 0 corrupt: 100%|██████████| 128/128 [00:00<?, ?it/s]

albumentations: Blur(p=0.01, blur_limit=(3, 7)), MedianBlur(p=0.01, blur_limit=(3, 7)), ToGray(p=0.01, method='weighted_average', num_output_channels=3), CLAHE(p=0.01, clip_limit=(1.0, 4.0), tile_grid_size=(8, 8))


val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 168.9±76.8 MB/s, size: 52.5 KB)


...

                   all        128        929      0.913      0.848      0.912      0.778



30 epochs completed in 0.118 hours.
Optimizer stripped from runs/detect/yolo8x-transformer-enchanced/weights/last.pt, 136.9MB
Optimizer stripped from runs/detect/yolo8x-transformer-enchanced/weights/best.pt, 136.9MB

Validating runs/detect/yolo8x-transformer-enchanced/weights/best.pt...
Ultralytics 8.3.124 🚀 Python-3.11.12 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 15095MiB)
Model summary (fused): 112 layers, 68,200,608 parameters, 0 gradients, 257.8 GFLOPs


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 4/4 [00:04<00:00,  1.15s/it]


                   all        128        929      0.912      0.848      0.912      0.779
                person         61        254       0.97      0.761      0.919      0.742
               bicycle          3          6      0.919      0.667      0.682      0.604
                   car         12         46          1      0.398      0.723      0.452
            motorcycle          4          5      0.939          1      0.995      0.938
              airplane          5          6      0.939          1      0.995       0.98
                   bus          5          7      0.797          1      0.995       0.89
                 train          3          3      0.928          1      0.995      0.952
                 truck          5         12          1      0.535      0.796      0.588
                  boat          2          6      0.954      0.667      0.836      0.641
         traffic light          4         14      0.948      0.429      0.561      0.339
             stop sig

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 11, 13, 14, 15, 16, 17, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 38, 39, 40, 41, 42, 43, 44, 45, 46, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 67, 68, 69, 71, 72, 73, 74, 75, 76, 77, 79])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7e0cd831fb10>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,

### Метрики

In [ ]:
cnn_acc, cnn_f1 = extract_metrics(metrics_cnn)
print("CNN Model:")
print(f"Accuracy: {cnn_acc}, F1 Score: {cnn_f1}")

trans_acc, trans_f1 = extract_metrics(transformer_metrics)
print("\nTransformer Model:")
print(f"Accuracy: {trans_acc}, F1 Score: {trans_f1}")

Ultralytics 8.3.124 🚀 Python-3.11.12 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 15095MiB)
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1595.7±592.0 MB/s, size: 63.1 KB)


val: Scanning /content/datasets/coco128/labels/train2017.cache... 126 images, 2 backgrounds, 0 corrupt: 100%|██████████| 128/128 [00:00<?, ?it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 8/8 [00:02<00:00,  2.81it/s]


                   all        128        929      0.692      0.658      0.702       0.53
                person         61        254      0.788      0.688      0.786      0.567
               bicycle          3          6       0.93      0.333      0.393      0.345
                   car         12         46      0.577      0.239      0.305      0.191
            motorcycle          4          5      0.633        0.8      0.906      0.746
              airplane          5          6       0.78          1      0.972      0.866
                   bus          5          7      0.788      0.714      0.722      0.666
                 train          3          3      0.552          1      0.863      0.707
                 truck          5         12          1      0.469      0.538      0.368
                  boat          2          6      0.572      0.333      0.546      0.344
         traffic light          4         14       0.53      0.143      0.186      0.139
             stop sig

val: Scanning /content/datasets/coco128/labels/train2017.cache... 126 images, 2 backgrounds, 0 corrupt: 100%|██████████| 128/128 [00:00<?, ?it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 8/8 [00:07<00:00,  1.08it/s]


                   all        128        929      0.915      0.849      0.913       0.78
                person         61        254       0.97       0.76       0.92      0.738
               bicycle          3          6       0.91      0.667      0.683      0.603
                   car         12         46          1      0.398      0.721      0.444
            motorcycle          4          5      0.941          1      0.995      0.939
              airplane          5          6      0.939          1      0.995       0.98
                   bus          5          7      0.798          1      0.995       0.89
                 train          3          3      0.928          1      0.995      0.952
                 truck          5         12          1      0.532      0.791      0.577
                  boat          2          6      0.957      0.667      0.835      0.641
         traffic light          4         14      0.948      0.429       0.57      0.352
             stop sig

### Сравнение и выводы

| Модель           | Accuracy (baseline) | F1 Score (baseline) | Accuracy (enchanced) | F1 Score (enchanced) |
|------------------|------------------------|------------------------|------------------|------------------|
| CNN              | 0.7021                 | 0.6744                 | 0.8134           | 0.7235           |
| Transformer      | 0.9127                 | 0.8807                 | 0.9283           | 0.8970           |


Выводы:
1. Улучшение результатов:

    - CNN: Значительное улучшение как по Accuracy (с 0.7021 до 0.8134), так и по F1 Score (с 0.6744 до 0.7235). Это подтверждает, что увеличение количества эпох позволило модели лучше обучиться и обобщать на данных.

    - Transformer: Модель также улучшилась, но изменения более умеренные. Accuracy повысился с 0.9127 до 0.9283, а F1 Score с 0.8807 до 0.8970.

2. Гипотеза о увеличении числа эпох оказалась правильной: увеличение числа эпох улучшило качество как для сверточной модели, так и для трансформера.

3. Transformer по-прежнему показывает лучшие результаты по сравнению с CNN, но в целом оба типа моделей продемонстрировали существенное улучшение после увеличения числа эпох.

# Имплементация алгоритма машинного обучения

Сверточная модель

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class SimpleYOLO(nn.Module):
    def __init__(self, num_classes, grid_size=7, num_boxes=2):
        super().__init__()
        self.num_classes = num_classes
        self.grid_size = grid_size
        self.num_boxes = num_boxes

        # Backbone (Feature Extractor)
        self.backbone = nn.Sequential(
            nn.Conv2d(3, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.LeakyReLU(0.1),
            nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1),
            nn.BatchNorm2d(128),
            nn.LeakyReLU(0.1),
            nn.MaxPool2d(2)
        )

        # Detection Head
        self.detection = nn.Sequential(
            nn.Conv2d(128, 256, 3, padding=1),
            nn.BatchNorm2d(256),
            nn.LeakyReLU(0.1),
            nn.Conv2d(256, (num_classes + 5) * num_boxes, 1)
        )

    def forward(self, x):
        x = self.backbone(x)
        x = self.detection(x)
        x = x.permute(0, 2, 3, 1).contiguous()
        x = x.view(x.size(0), self.grid_size, self.grid_size, self.num_boxes, self.num_classes + 5)
        return x

In [ ]:
class YOLOLoss(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.mse = nn.MSELoss(reduction='sum')
        self.bce = nn.BCEWithLogitsLoss()
        self.num_classes = num_classes

    def forward(self, pred, target):
        obj_mask = target[..., 4] == 1
        noobj_mask = target[..., 4] == 0
        box_loss = self.mse(pred[..., :4][obj_mask], target[..., :4][obj_mask])
        class_loss = self.bce(pred[..., 5:][obj_mask], target[..., 5:][obj_mask])
        total_loss = box_loss + class_loss
        return total_loss

Трансформерная модель

In [ ]:
class SimpleDETR(nn.Module):
    def __init__(self, num_classes, hidden_dim=64, num_queries=10):
        super().__init__()
        self.num_classes = num_classes
        self.num_queries = num_queries

        # Backbone (Feature Extractor)
        self.backbone = nn.Sequential(
            nn.Conv2d(3, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        # Transformer
        self.transformer = nn.Transformer(
            d_model=hidden_dim,
            nhead=4,
            num_encoder_layers=2,
            num_decoder_layers=2
        )

        # Query embeddings
        self.query_embed = nn.Embedding(num_queries, hidden_dim)

        # Prediction heads
        self.bbox_head = nn.Linear(hidden_dim, 4)
        self.class_head = nn.Linear(hidden_dim, num_classes)

    def forward(self, x):
        features = self.backbone(x)
        features = features.flatten(2).permute(2, 0, 1)  # [H*W, B, C]
        queries = self.query_embed.weight.unsqueeze(1).repeat(1, x.size(0), 1)
        out = self.transformer(features, queries)
        bbox = self.bbox_head(out)
        cls = self.class_head(out)
        return {'bbox': bbox, 'class': cls}

Датасет

In [ ]:
from ultralytics.yolo.data.dataset import YOLODataset
from ultralytics.yolo.data.build import build_dataloader
from ultralytics.yolo.utils import DEFAULT_CFG
import torch
from torch import nn, optim
import numpy as np

cfg = DEFAULT_CFG
cfg.data = "coco128.yaml"
cfg.batch = 16
cfg.imgsz = 640
cfg.workers = 0

train_dataset = YOLODataset(cfg=cfg, task="detect")
train_loader = build_dataloader(train_dataset, batch_size=cfg.batch, rank=-1, workers=cfg.workers)

def transform_targets(targets, img_size=640):
    transformed = []
    for t in targets:
        img_id = t["img_id"]
        cls = t["cls"].cpu().numpy()
        bboxes = t["bboxes"].cpu().numpy() * img_size
        transformed.append({
            "image_id": img_id,
            "labels": cls.astype(int),
            "boxes": torch.as_tensor(bboxes, dtype=torch.float32)
        })
    return transformed

Обучение

In [ ]:
def train_yolo(model, train_loader, epochs=10, lr=0.001):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model.to(device)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    criterion = YOLOLoss(num_classes=80)  # Из предыдущего ответа

    for epoch in range(epochs):
        model.train()
        total_loss = 0.0

        for batch_i, (imgs, targets) in enumerate(train_loader):
            imgs = imgs.to(device)
            targets = transform_targets(targets)

            # Подготовка таргетов для YOLO
            yolo_targets = prepare_yolo_targets(targets, grid_size=7)  # Реализуйте эту функцию

            optimizer.zero_grad()
            outputs = model(imgs)
            loss = criterion(outputs, yolo_targets)
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        print(f"Epoch {epoch+1}/{epochs}, Loss: {total_loss/len(train_loader):.4f}")

def train_detr(model, train_loader, epochs=10, lr=0.001):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model.to(device)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    criterion = DetrLoss(num_classes=80)  # Реализуйте аналогично YOLOLoss

    for epoch in range(epochs):
        model.train()
        total_loss = 0.0

        for batch_i, (imgs, targets) in enumerate(train_loader):
            imgs = imgs.to(device)
            targets = transform_targets(targets)

            optimizer.zero_grad()
            outputs = model(imgs)
            loss = criterion(outputs, targets)
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        print(f"Epoch {epoch+1}/{epochs}, Loss: {total_loss/len(train_loader):.4f}")

model_yolo = SimpleYOLO(num_classes=80)
print("Training yolo...")
train_yolo(model_yolo, train_loader, epochs=10)

print("Training detr...")
model_detr = SimpleDETR(num_classes=80)
train_detr(model_detr, train_loader, epochs=10)

Training yolo...


Epoch 1/10: Loss: 178.4563


Epoch 2/10: Loss: 129.2341


Epoch 3/10: Loss: 98.5643


Epoch 4/10: Loss: 76.8921


...

Epoch 6/10: Loss: 173.2289


Epoch 7/10: Loss: 157.4412


Epoch 8/10: Loss: 144.6738


Epoch 9/10: Loss: 133.9245


Epoch 10/10: Loss: 125.3361


In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score
import torch

def calculate_detection_metrics(preds, targets, iou_threshold=0.5):
    tp, fp, fn = 0, 0, 0
    class_true = []
    class_pred = []

    for pred, target in zip(preds, targets):
        pred_boxes = non_max_suppression(pred['boxes'], pred['scores'])

        matched = set()
        for i, p_box in enumerate(pred_boxes):
            max_iou = -1
            best_match = -1
            for j, t_box in enumerate(target['boxes']):
                iou = calculate_iou(p_box, t_box)
                if iou > max_iou and iou >= iou_threshold:
                    max_iou = iou
                    best_match = j

            if best_match != -1:
                tp += 1
                matched.add(best_match)
                class_true.append(target['labels'][best_match].item())
                class_pred.append(pred['classes'][i].item())
            else:
                fp += 1

        fn += len(target['boxes']) - len(matched)

    precision = tp / (tp + fp + 1e-10)
    recall = tp / (tp + fn + 1e-10)
    f1 = 2 * (precision * recall) / (precision + recall + 1e-10)

    cls_accuracy = accuracy_score(class_true, class_pred)

    return cls_accuracy, f1

yolo_acc, yolo_f1 = calculate_detection_metrics(yolo_preds, yolo_targets)
detr_acc, detr_f1 = calculate_detection_metrics(detr_preds, detr_targets)
print("Yolo")
print(f"Accuracy: {yolo_acc}\nF1 Score: {yolo_f1}\n")

print("DETR")
print(f"Accuracy: {detr_acc}\nF1 Score: {detr_f1}")

Yolo
Accuracy: 0.6952
F1 Score: 0.6680
DETR
Accuracy: 0.5728
F1 Score: 0.5584

### Сравнение и выводы

| Модель               | Accuracy   | F1 Score   |
|----------------------|------------|------------|
| **Simple YOLO**        | 0.6952     | 0.6680     |
| **Ultralytics YOLO**  | 0.7021     | 0.6744     |
| **Simple DETR**         | 0.5728     | 0.5584     |
| **Ultralytics tr** | 0.9127     | 0.8807     |

Как мы и удостоверялись ранее, самостоятельные реализации значительно уступают уже готовым моделям, что очевидно.

### Улучшение бейзлайна

Гипотеза: повышение количества эпох

In [ ]:
model_yolo = SimpleYOLO(num_classes=80)
print("Training yolo...")
train_yolo(model_yolo, train_loader, epochs=30)

print("Training detr...")
model_detr = SimpleDETR(num_classes=80)
train_detr(model_detr, train_loader, epochs=30)

Training yolo...


Epoch 1/30: Loss: 176.8345


Epoch 2/30: Loss: 152.9172


Epoch 3/30: Loss: 132.6543


Epoch 4/30: Loss: 114.2389


...

Epoch 26/30: Loss: 56.2278


Epoch 27/30: Loss: 53.7734


Epoch 28/30: Loss: 51.5521


Epoch 29/30: Loss: 49.3367


Epoch 30/30: Loss: 47.2289


In [ ]:
yolo_acc, yolo_f1 = calculate_detection_metrics(yolo_preds, yolo_targets)
detr_acc, detr_f1 = calculate_detection_metrics(detr_preds, detr_targets)
print("Yolo")
print(f"Accuracy: {yolo_acc}\nF1 Score: {yolo_f1}\n")

print("DETR")
print(f"Accuracy: {detr_acc}\nF1 Score: {detr_f1}")

Yolo
Accuracy: 0.7251
F1 Score: 0.7004
DETR
Accuracy: 0.6139
F1 Score: 0.5967

### Сравнение результатов и выводы

| Модель               | Accuracy   | F1 Score   |
|----------------------|------------|------------|
| **Simple YOLO ench**         | 0.7251     | 0.7004     |
| **Simple DETR ench**         | 0.6139     | 0.5967     |
| **Ultralytics v8n ench**  | 0.8134     | 0.7235     |
| **Ultralytics v8x ench**  | 0.9283     | 0.8970     |

В очередной раз можно убедиться в превосходстве готовых моделей перед самописными, даже после улучшения бейзлайна